# Phase 3 Experiments - Phase C+D: Gradual Pruning

This notebook runs **Gradual pruning** (with cubic schedule) on all 4 KD models.

**Runs**: 4.5, 4.6, 4.7, 4.8

**eval_fold per KD model**:
- KD1, KD2 (Teacher T1): eval_fold=3
- KD3, KD4 (Teacher T2): eval_fold=2

**DEPENDENCY**: Phase B must complete first and upload KD models to HuggingFace!

**Total Runs**: 4

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q iterative-stratification scikit-learn pandas numpy tqdm

In [ ]:
# Setup logging
import sys
from datetime import datetime

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger("/kaggle/working/experiment_log_gradual.txt")
print(f"Experiment started at: {datetime.now()}")
print("Phase C+D: Gradual Pruning on KD Models")

In [ ]:
# Clone repository
!git clone https://github.com/SaifSiddique009/kd_pruning_quantization_framework_for_nlp.git
%cd kd_pruning_quantization_framework_for_nlp
!git checkout phase3-comprehensive-experiments
!git log -1 --oneline

In [ ]:
# KD Model paths from Phase B (uploaded to HuggingFace)
AUTHOR = "Saif-Siddique"

KD_MODELS = {
    "KD1": f"{AUTHOR}/bangla-cyberbully-kd1-xlmroberta-to-sahajbert",
    "KD2": f"{AUTHOR}/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small",
    "KD3": f"{AUTHOR}/bangla-cyberbully-kd3-banglabert-to-sahajbert",
    "KD4": f"{AUTHOR}/bangla-cyberbully-kd4-banglabert-to-banglabert-small",
}

print("KD Models to be used (from HuggingFace):")
for kd_id, path in KD_MODELS.items():
    print(f"  {kd_id}: {path}")

In [ ]:
# Verify KD models are accessible
from transformers import AutoModel

print("Verifying KD models are accessible...")
for kd_id, path in KD_MODELS.items():
    try:
        _ = AutoModel.from_pretrained(path)
        print(f"  [OK] {kd_id}: {path}")
        del _
    except Exception as e:
        print(f"  [ERROR] {kd_id}: {e}")
        print(f"  Make sure Phase B completed and uploaded models!")

---
## Gradual Pruning Runs (4.5-4.8)
---

### Run 4.5: KD1 + Gradual Pruning (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD1_gradual

### Run 4.6: KD2 + Gradual Pruning (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD2_gradual

### Run 4.7: KD3 + Gradual Pruning (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd3-banglabert-to-sahajbert" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD3_gradual

### Run 4.8: KD4 + Gradual Pruning (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd4-banglabert-to-banglabert-small" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD4_gradual

---
## Final Status Check
---

In [ ]:
import os
import json
from datetime import datetime

print(f"\n{'='*70}")
print(f"GRADUAL PRUNING COMPLETION STATUS - {datetime.now()}")
print(f"{'='*70}\n")

experiments = [
    ("4.5 KD1 + Gradual", "./results/scenario4/KD1_gradual", 3),
    ("4.6 KD2 + Gradual", "./results/scenario4/KD2_gradual", 3),
    ("4.7 KD3 + Gradual", "./results/scenario4/KD3_gradual", 2),
    ("4.8 KD4 + Gradual", "./results/scenario4/KD4_gradual", 2),
]

success_count = 0

print(f"{'Experiment':<25} {'Fold':<6} {'F1 Weighted':<12} {'F1 Macro':<12} {'Status'}")
print("-" * 70)

for name, output_dir, fold in experiments:
    json_path = os.path.join(output_dir, "results_final.json")
    
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        
        if isinstance(data, list):
            final_metrics = data[-1]
        else:
            final_metrics = data
        
        f1_weighted = final_metrics.get('f1_weighted', 'N/A')
        f1_macro = final_metrics.get('f1_macro', 'N/A')
        
        print(f"{name:<25} {fold:<6} {f1_weighted:<12.4f} {f1_macro:<12.4f} SUCCESS")
        success_count += 1
    else:
        print(f"{name:<25} {fold:<6} {'N/A':<12} {'N/A':<12} FAILED")

print("-" * 70)
print(f"\nCompleted: {success_count}/{len(experiments)} experiments")
print(f"{'='*70}")

In [ ]:
# Copy results to output
!cp -r ./results /kaggle/working/
!ls -la /kaggle/working/results/scenario4/

In [ ]:
print(f"\nGradual Pruning notebook completed at: {datetime.now()}")
print("\nDownload results from /kaggle/working/results/scenario4/")